# Pre-processing MultiplEYE Data

This notebook provides a step-by-step guide through how to process the eye-tracking data and the psychometric tests data collected within the MultiplEYE project. This goal of this notebook is twofold:

1. To provide a step-by-step guide on how to preprocess MultiplEYE data using the `pymovements` library and our custom preprocessing functions.
2. To serve as a tutorial for researchers who want to preprocess their own MultiplEYE data, or data from other eye-tracking datasets, using the `pymovements` library.

## Preparation steps
1. Download the data folder from the online repository. Note that this is only possible if you have access to at least one data collection protected folder. You will have access if you are an active member of one data collection group. Download the entire content of the folder.
When you download it from SwitchDrive, it will automatically create a .tar file.
2. Add the folder to the `data/` folder in this repo. The name of the folder is the data collection name, e.g., `MultiplEYE_ZH_CH_Zurich_1_2025`.
3. Extract the .tar file in the `data/` folder.
4. Make sure that the folder structure is correct. It should look like the one online and like this (there might be more data but this is not relevant at this point):
```
	MultiplEYE_ZH_CH_Zurich_1_2025/
		documentation/
		eye-tracking-sessions/
			001_.../
			002_.../
			...
			pilot_sessions/
				001_.../
				002_.../
				...
		psychometric-tests-sessions/
		stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/
		...
```

## The config file



The pipeline uses a config file which can be used to specify parameters and settings for the preprocessing. It is typically named `multipleye_settings_preprocessing.yaml`. You can load it explicitly or rely on the default loading mechanism (CWD, environment variable, or legacy root).

Once you have your config file ready, you can load it as shown below.

In [39]:
# from preprocessing.data_collection.multipleye_data_collection import prepare_language_folder
from preprocessing.data_collection.multipleye_data_collection import (
    MultipleyeDataCollection,
)

import preprocessing

# the settings will be loaded into general config module, so we can access all settings at the same place
from preprocessing import settings

from preprocessing.scripts.prepare_language_folder import prepare_language_folder
from preprocessing.metrics.reading.words import all_tokens_from_aois
from pymovements.measure.reading.processing import compute_reading_measures

import polars as pl

In [ ]:
# If you have a specific config file, load it here:
# settings.load_from_yaml("data/MultiplEYE_<...>/multipleye_settings_preprocessing.yaml")

In [2]:
# get the data collection name from the settings and create the path to the data folder
print(f"Active Data Collection: {settings.DATA_COLLECTION_NAME}")
print(f"Dataset Directory: {settings.DATASET_DIR}")

Active Data Collection: MultiplEYE_SV_CH_Zurich_1_2026
Dataset Directory: /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/data/MultiplEYE_SV_CH_Zurich_1_2026


### Inspecting and overriding configuration

After loading the config, you can inspect which sessions are included or excluded, and override these values for the current session without modifying the YAML file.

In [3]:
print(f"Include pilots:  {settings.INCLUDE_PILOTS}")
print(f"Included:        {settings.INCLUDE_SESSIONS}")
print(f"Excluded:        {settings.EXCLUDE_SESSIONS}")
print(f"Output dir:      {settings.OUTPUT_DIR}")
print(f"Run preflight:   {settings.RUN_PREFLIGHT_CHECK}")
print(f"Overwrite:       {settings.OVERWRITE}")

# Override example (uncomment to limit processing to specific sessions):
# settings.INCLUDE_SESSIONS = ["014_DE_DE_1_ET1", "023_DE_DE_1_ET1"]
# settings.EXCLUDE_SESSIONS = []

Include pilots:  True
Included:        ['009_SV_CH_1_ET1']
Excluded:        []
Output dir:      /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026
Run preflight:   True
Overwrite:       True


## MultiplEYE-specific preprocessing & cleaning

In order to be able to run a more generic preprocessing, the MultiplEYE data folder for one language needs to be cleaned and organized in a specific way. Running the script below will:
- unzip session folders if needed
- move session folders from core_sessions folder to the top folder
- check if there is a config file in the stimuli folder (if not, the stimulus folder was probably not uploaded correctly)
- check if there are psychometric tests (if applicable)
	- if necessary, restructure the psychometric test folder.

These steps are very individual for this data collection and results from bugs or changes across the years of collecting data.

Note that executing the cell below for the first time can take very long. However, it will run through quickly after this initial run.

In [4]:
# run the preparation function to prepare the language folder structure
prepare_language_folder()

2026-08-04 09:42:47,169 - preprocessing - INFO - Copying stimulus assets to /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026...


Next, we create a `MultipleyeDataCollection` object from the data folder. This will allow us to easily access the sessions and their information in the next steps.

In [5]:
multipleye = MultipleyeDataCollection.create_from_data_folder(
    settings.DATASET_DIR,
    include_pilots=settings.INCLUDE_PILOTS,
    excluded_sessions=settings.EXCLUDE_SESSIONS,
    included_sessions=settings.INCLUDE_SESSIONS,
)

2026-08-04 09:42:50,319 - preprocessing - INFO - Lab config loaded from /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/config/config_sv_ch_Zurich_1_2026.py
2026-08-04 09:42:50,321 - preprocessing - INFO - JSON lab config loaded from /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/config/MultiplEYE_SV_CH_Zurich_1_2026_lab_configuration.json
2026-08-04 09:42:50,324 - preprocessing - INFO - MultipleyeDataCollection initialized. data_root: /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/data/MultiplEYE_SV_CH_Zurich_1_2026/eye-tracking-sessions
2026-08-04 09:42:50,326 - preprocessing - INFO - Main config loaded from /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/config/config_sv_ch_Zurich_1.py
2026

### Preflight check

Before processing, run a preflight check to validate the dataset structure and catch common issues (missing files, incorrect folder layout, etc.). In case EDF files are missing, you can use `settings.EXCLUDE_SESSIONS = []` to exclude specific sessions, as shown a few cells above.

In [6]:
preprocessing.run_preflight_check(multipleye)

2026-08-04 09:42:59,248 - preprocessing - INFO - 
  Preflight check — all input files found


## Stage 0: Converting EDF to ASC and Preparing Session-Level Information

Stage 0 refers to the initial steps of preprocessing, which involve converting raw eye-tracking data from its original format (e.g., EDF) into a more accessible format (e.g., ASC), and preparing session-level information. This stage is specific to EyeLink eye-trackers and can be omitted for other eye-trackers.

In [7]:
multipleye.convert_edf_to_asc()

2026-08-04 09:45:52,701 - preprocessing - INFO - Starting EDF to ASC conversion for 1 sessions.
Converting EDF to ASC: 100%|██████████| 1/1 [00:00<00:00, 618.63it/s]
2026-08-04 09:45:52,713 - preprocessing - INFO - EDF to ASC conversion completed.


Once this conversion has been completed, we can load all sessions and parse the .asc files.

In [8]:
multipleye.prepare_session_level_information()

Preparing session 009_SV_CH_1_ET1: 100%|██████████| 1/1 [00:09<00:00,  9.51s/it]


In [9]:
# print an overview on the data collection and the sessions
multipleye

Title	MultiplEYE_SV_CH_Zurich_1_2026
Dataset_type	MultiplEYE
Number_of_sessions	0
Number_of_pilots	1
Tested_language	SV
Country	CH
Year	2026
Number of eye-tracking (ET) sessions per participant	1

## Stage 1: Extracting Gaze Samples

In the first preprocessing stage, we extract gaze samples from the .asc files and create a gaze dataframe for each session. This dataframe contains the raw gaze data, including the x and y coordinates of the gaze, the timestamp. We also save the raw gaze data in a separate file for each session.

The next steps are performed for one session only. It is always possible to loop over all sessions and apply the same preprocessing steps to each of them, but for the sake of clarity and simplicity, we will work with one session as an example.



In [10]:
# pick only one session as an example to work with in the next steps
sessions = list(multipleye)  # list of Session objects
sess = sessions[0]  # a real Session
sid = sess.sid  # get the session ID (Sid) from the Session
sid

Sid(pid='009', lang='SV', country='CH', lab='1', session='ET1', session_id=1, postfix='')

In [11]:
type(sess)

preprocessing.data_collection.session.Session

### Creating Gaze Frame from ASCII File

In [12]:
gaze = preprocessing.load_gaze_data(
    asc_file=sess.asc_path,
    lab_config=sess.lab_config,
    sid=sess.sid,
    trial_cols=settings.TRIAL_COLS,
    messages=settings.ANSWER_MSG_PATTERNS,  # for comprehension questions later - see Stage 4.
)

In [13]:
# save gaze and metadata
preprocessing.save_raw_data(sid, gaze)
preprocessing.save_session_metadata(sid, gaze)

In [14]:
sid.raw_data_dir, sid.metadata_dir

(PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/raw_data/009_SV_CH_1_ET1'),
 PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/metadata/009_SV_CH_1_ET1'))

### Output directory structure

All preprocessed data is organised by data type under `preprocessed_data/<data_collection_name>/`. Each data type folder contains one subfolder per session:

```
preprocessed_data/<dcn>/
├── raw_data/
│   └── <session_save_name>/
├── fixations/
│   └── <session_save_name>/
├── saccades/
│   └── <session_save_name>/
├── scanpaths/
│   └── <session_save_name>/
├── reading_measures/
│   └── <session_save_name>/
├── sanity_checks/
│   └── <session_save_name>/
├── metadata/
│   └── <session_save_name>/
│       ├── gaze_metadata.json
│       ├── experiment.yaml
│       ├── calibrations.tsv
│       ├── calibrations.feather
│       ├── validations.tsv
│       ├── validations.feather
│       └── <session_idf>_overview.yaml
├── participant_data.csv
├── <dcn>_overview.yaml
└── stimuli_<dcn>/
```

The `Sid` object provides convenient properties to access each path:

In [15]:
print(f"Raw data:        {sid.raw_data_dir}")
print(f"Metadata:        {sid.metadata_dir}")
print(f"Fixations:       {sid.fixations_dir}")
print(f"Saccades:        {sid.saccades_dir}")
print(f"Scanpaths:       {sid.scanpaths_dir}")
print(f"Reading measures: {sid.reading_measures_dir}")

Raw data:        /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/raw_data/009_SV_CH_1_ET1
Metadata:        /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/metadata/009_SV_CH_1_ET1
Fixations:       /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/fixations/009_SV_CH_1_ET1
Saccades:        /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/saccades/009_SV_CH_1_ET1
Scanpaths:       /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/scanpaths/009_SV_CH_1_ET1
Reading measures: /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/reading_measures/009_SV_CH_1_ET1


In order to have the metadata which is extracted by pymovements available to create out session overview, we get this information from pymovements and store it in our session object.

In [16]:
sess.pm_gaze_metadata = gaze._metadata
sess.calibrations = gaze.calibrations
sess.validations = gaze.validations

### Coordinate and Velocity Preprocessing

Eye movements are recorded in screen pixel coordinates, which depend on stimulus size and monitor setup. To compare gaze behavior across participants, screens, or datasets, it is standard to convert pixel positions 
into **degrees of visual angle (dva)**. Next, we compute **gaze velocity**, which allows us to detect saccades and distinguish them from fixations.

In [17]:
# inspect the gaze samples
gaze.samples.head()

time,pupil,stimulus,activity,practice,trial,page,session,pixel
i64,f64,str,str,bool,str,str,str,list[f64]
2170575,652.0,"""Enc_WikiMoon_13""","""reading""",true,"""PRACTICE_trial_1""","""page_1""","""009_SV_CH_1_ET1""","[80.5, 103.7]"
2170576,653.0,"""Enc_WikiMoon_13""","""reading""",true,"""PRACTICE_trial_1""","""page_1""","""009_SV_CH_1_ET1""","[81.3, 103.2]"
2170577,642.0,"""Enc_WikiMoon_13""","""reading""",true,"""PRACTICE_trial_1""","""page_1""","""009_SV_CH_1_ET1""","[78.3, 105.9]"
2170578,645.0,"""Enc_WikiMoon_13""","""reading""",true,"""PRACTICE_trial_1""","""page_1""","""009_SV_CH_1_ET1""","[80.5, 106.8]"
2170579,651.0,"""Enc_WikiMoon_13""","""reading""",true,"""PRACTICE_trial_1""","""page_1""","""009_SV_CH_1_ET1""","[78.8, 104.4]"


In [18]:
preprocessing.preprocess_gaze(gaze)

In [19]:
# inspect the preprocessed gaze samples, the dataframe should now also contain a position in dva and velocity columns
gaze.samples.head()

time,pupil,stimulus,activity,practice,trial,page,session,pixel,position,velocity
i64,f64,str,str,bool,str,str,str,list[f64],list[f64],list[f64]
2170575,652.0,"""Enc_WikiMoon_13""","""reading""",true,"""PRACTICE_trial_1""","""page_1""","""009_SV_CH_1_ET1""","[80.5, 103.7]","[-15.111077, -10.470344]","[-0.544006, 2.03784]"
2170576,653.0,"""Enc_WikiMoon_13""","""reading""",true,"""PRACTICE_trial_1""","""page_1""","""009_SV_CH_1_ET1""","[81.3, 103.2]","[-15.090872, -10.483245]","[-0.415798, 1.847938]"
2170577,642.0,"""Enc_WikiMoon_13""","""reading""",true,"""PRACTICE_trial_1""","""page_1""","""009_SV_CH_1_ET1""","[78.3, 105.9]","[-15.166621, -10.413566]","[-0.439114, 1.478056]"
2170578,645.0,"""Enc_WikiMoon_13""","""reading""",true,"""PRACTICE_trial_1""","""page_1""","""009_SV_CH_1_ET1""","[80.5, 106.8]","[-15.111077, -10.390333]","[-0.545122, 1.40235]"
2170579,651.0,"""Enc_WikiMoon_13""","""reading""",true,"""PRACTICE_trial_1""","""page_1""","""009_SV_CH_1_ET1""","[78.8, 104.4]","[-15.154, -10.452281]","[-0.490519, 0.990696]"


Save our events data.

In [20]:
gaze = preprocessing.load_trial_level_events_data(
    gaze,
    sess.sid,
    event_type=settings.FIXATION,
    file_pattern=None,
)

gaze = preprocessing.load_trial_level_events_data(
    gaze,
    sess.sid,
    event_type=settings.SACCADE,
    file_pattern=None,
)

## Stage 2b: Map Fixations to AOIs

Once we have the fixations, we can map each of them to the AOIs of the stimulus. The resulting scanpath can then be saved. Note that this features is not yet completely finished.

In [21]:
preprocessing.map_fixations_to_aois(gaze, sess.stimuli)

In [22]:
# The resulting mapping can be stored as a scanpath, which is a sequence of AOIs that were fixated in the order they were fixated.
preprocessing.save_scanpaths(sid, gaze)

In [23]:
# save metadata again
preprocessing.save_session_metadata(sid, gaze)

## NEW

In [24]:
group_columns = [settings.TRIAL_COL, settings.STIMULUS_COL, settings.PAGE_COL]

only_fix = (
    gaze.events.frame.filter(
        (pl.col("name") == settings.FIXATION)
        & (pl.col(settings.WORD_IDX_COL).is_not_null())
    )
    .with_row_count("fixation_id")
    .sort(group_columns + ["onset"])
)
rm_all_trials = []

2026-08-04 09:57:15,510 - py.warnings - PYWARN:WARNING - /tmp/ipykernel_11850/465983878.py:8: DeprecationWarning: `DataFrame.with_row_count` is deprecated; use `with_row_index` instead. Note that the default column name has changed from 'row_nr' to 'index'.
  .with_row_count("fixation_id")



In [37]:
stim = sess.stimuli[0]
aois = stim.text_stimulus.aois
words_only = all_tokens_from_aois(aois, trial=stim.trial_id)
words_only = words_only.with_columns(
    pl.lit(stim.name).alias(settings.STIMULUS_COL)
)
trial_idx = stim.trial_id

In [47]:
trial_fix = only_fix.filter((pl.col(settings.TRIAL_COL) == trial_idx))

In [56]:
grouped_fix = trial_fix.group_by(settings.PAGE_COL)

In [57]:
grouped_fix.head()

page,fixation_id,trial,stimulus,name,onset,offset,duration,location_x,location_y,amplitude,peak_velocity,dispersion,char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,word_idx,word_idx_in_line,word
str,u32,str,str,str,i64,i64,i64,f64,f64,f64,f64,str,i64,str,f64,f64,i64,f64,i64,i64,i64,i64,str
"""page_5""",617,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8099279,null,342,522.026822,118.760641,null,null,null,31,"""d""",515.0,89.0,14,31.0,31,0,5,5,"""bildigenkänning"""
"""page_5""",618,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8099775,null,371,535.657527,118.296237,null,null,null,32,"""i""",529.0,89.0,14,31.0,32,0,5,5,"""bildigenkänning"""
"""page_5""",619,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8100195,null,113,662.557018,110.981579,null,null,null,41,"""n""",655.0,89.0,14,31.0,41,0,5,5,"""bildigenkänning"""
"""page_5""",620,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8100358,null,104,821.212381,115.624762,null,null,null,52,"""t""",809.0,89.0,14,31.0,52,0,7,7,"""eye-trackern"""
"""page_5""",621,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8100705,null,154,734.968387,118.576774,null,null,null,46,"""n""",725.0,89.0,14,31.0,46,0,6,6,"""kan"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""page_1""",410,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8010272,null,124,205.8472,191.1304,null,null,null,28,"""M""",193.0,149.45,14,89.9,8,1,2,1,"""”MultiplEYE”"""
"""page_1""",411,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8010440,null,198,194.563819,112.551759,null,null,null,8,"""Y""",193.0,89.0,14,31.0,8,0,0,0,"""MultiplEYE-projektet"""
"""page_1""",412,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8010997,null,271,135.190441,222.323529,null,null,null,23,"""n""",123.0,149.45,14,89.9,3,1,1,0,"""Namnet"""


In [59]:
settings.PAGE_PREFIX

'page_'

In [58]:
grouped_fix.head()

page,fixation_id,trial,stimulus,name,onset,offset,duration,location_x,location_y,amplitude,peak_velocity,dispersion,char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,word_idx,word_idx_in_line,word
str,u32,str,str,str,i64,i64,i64,f64,f64,f64,f64,str,i64,str,f64,f64,i64,f64,i64,i64,i64,i64,str
"""page_8""",799,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8171124,null,249,314.6948,112.0636,null,null,null,16,"""o""",305.0,89.0,14,31.0,16,0,1,1,"""bakom"""
"""page_8""",800,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8171554,null,188,594.6,110.551323,null,null,null,36,""" """,585.0,89.0,14,31.0,36,0,4,4,"""att"""
"""page_8""",801,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8171790,null,120,712.319835,112.14876,null,null,null,45,"""f""",711.0,89.0,14,31.0,45,0,6,6,"""fortfarande"""
"""page_8""",802,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8171960,null,241,871.241322,95.730579,null,null,null,56,"""n""",865.0,89.0,14,31.0,56,0,7,7,"""finns"""
"""page_8""",803,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8172247,null,106,974.480374,104.823364,null,null,null,63,"""l""",963.0,89.0,14,31.0,63,0,9,9,"""lite"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""page_5""",617,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8099279,null,342,522.026822,118.760641,null,null,null,31,"""d""",515.0,89.0,14,31.0,31,0,5,5,"""bildigenkänning"""
"""page_5""",618,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8099775,null,371,535.657527,118.296237,null,null,null,32,"""i""",529.0,89.0,14,31.0,32,0,5,5,"""bildigenkänning"""
"""page_5""",619,"""trial_10""","""PopSci_MultiplEYE_1""","""fixation""",8100195,null,113,662.557018,110.981579,null,null,null,41,"""n""",655.0,89.0,14,31.0,41,0,5,5,"""bildigenkänning"""


In [61]:
for page in grouped_fix:
    print(page)

(('page_10',), shape: (34, 24)
┌────────────┬──────────┬────────────┬─────────┬───┬──────────┬──────────┬────────────┬────────────┐
│ fixation_i ┆ trial    ┆ stimulus   ┆ page    ┆ … ┆ line_idx ┆ word_idx ┆ word_idx_i ┆ word       │
│ d          ┆ ---      ┆ ---        ┆ ---     ┆   ┆ ---      ┆ ---      ┆ n_line     ┆ ---        │
│ ---        ┆ str      ┆ str        ┆ str     ┆   ┆ i64      ┆ i64      ┆ ---        ┆ str        │
│ u32        ┆          ┆            ┆         ┆   ┆          ┆          ┆ i64        ┆            │
╞════════════╪══════════╪════════════╪═════════╪═══╪══════════╪══════════╪════════════╪════════════╡
│ 944        ┆ trial_10 ┆ PopSci_Mul ┆ page_10 ┆ … ┆ 0        ┆ 2        ┆ 2          ┆ eye-tracki │
│            ┆          ┆ tiplEYE_1  ┆         ┆   ┆          ┆          ┆            ┆ ngdata     │
│ 945        ┆ trial_10 ┆ PopSci_Mul ┆ page_10 ┆ … ┆ 0        ┆ 2        ┆ 2          ┆ eye-tracki │
│            ┆          ┆ tiplEYE_1  ┆         ┆   ┆        

In [41]:
aois.group_by(settings.PAGE_COL).head()

page,char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,word_idx,word_idx_in_line,word
str,i64,str,f64,f64,i64,f64,i64,i64,i64,i64,str
"""page_1""",0,"""M""",81.0,89.0,14,31.0,0,0,0,0,"""MultiplEYE-projektet"""
"""page_1""",1,"""u""",95.0,89.0,14,31.0,1,0,0,0,"""MultiplEYE-projektet"""
"""page_1""",2,"""l""",109.0,89.0,14,31.0,2,0,0,0,"""MultiplEYE-projektet"""
"""page_1""",3,"""t""",123.0,89.0,14,31.0,3,0,0,0,"""MultiplEYE-projektet"""
"""page_1""",4,"""i""",137.0,89.0,14,31.0,4,0,0,0,"""MultiplEYE-projektet"""
…,…,…,…,…,…,…,…,…,…,…,…
"""page_5""",0,"""M""",81.0,89.0,14,31.0,0,0,0,0,"""Med"""
"""page_5""",1,"""e""",95.0,89.0,14,31.0,1,0,0,0,"""Med"""
"""page_5""",2,"""d""",109.0,89.0,14,31.0,2,0,0,0,"""Med"""


In [ ]:
words_only

In [64]:
for fix in only_fix.group_by(group_columns):
    print(fix)

(('trial_6', 'PopSci_Caveman_12', 'page_4'), shape: (24, 24)
┌─────────────┬─────────┬─────────────┬────────┬───┬──────────┬──────────┬────────────┬────────────┐
│ fixation_id ┆ trial   ┆ stimulus    ┆ page   ┆ … ┆ line_idx ┆ word_idx ┆ word_idx_i ┆ word       │
│ ---         ┆ ---     ┆ ---         ┆ ---    ┆   ┆ ---      ┆ ---      ┆ n_line     ┆ ---        │
│ u32         ┆ str     ┆ str         ┆ str    ┆   ┆ i64      ┆ i64      ┆ ---        ┆ str        │
│             ┆         ┆             ┆        ┆   ┆          ┆          ┆ i64        ┆            │
╞═════════════╪═════════╪═════════════╪════════╪═══╪══════════╪══════════╪════════════╪════════════╡
│ 2274        ┆ trial_6 ┆ PopSci_Cave ┆ page_4 ┆ … ┆ 1        ┆ 15       ┆ 0          ┆ hitta”,    │
│             ┆         ┆ man_12      ┆        ┆   ┆          ┆          ┆            ┆            │
│ 2275        ┆ trial_6 ┆ PopSci_Cave ┆ page_4 ┆ … ┆ 1        ┆ 16       ┆ 1          ┆ säger      │
│             ┆         ┆ man_

In [80]:
for (trial_idx, stim_idx, page_idx), df in only_fix.group_by(group_columns):
    words_only = all_tokens_from_aois(df, trial=trial_idx)
    print(words_only.head())

shape: (5, 4)
┌──────────┬────────┬──────────┬────────────────────┐
│ trial    ┆ page   ┆ word_idx ┆ word               │
│ ---      ┆ ---    ┆ ---      ┆ ---                │
│ str      ┆ str    ┆ i64      ┆ str                │
╞══════════╪════════╪══════════╪════════════════════╡
│ trial_10 ┆ page_3 ┆ 3        ┆ stödja             │
│ trial_10 ┆ page_3 ┆ 7        ┆ stor               │
│ trial_10 ┆ page_3 ┆ 8        ┆ flerspråkig        │
│ trial_10 ┆ page_3 ┆ 9        ┆ eye-trackingkorpus │
│ trial_10 ┆ page_3 ┆ 11       ┆ göra               │
└──────────┴────────┴──────────┴────────────────────┘
shape: (5, 4)
┌─────────┬────────┬──────────┬───────────┐
│ trial   ┆ page   ┆ word_idx ┆ word      │
│ ---     ┆ ---    ┆ ---      ┆ ---       │
│ str     ┆ str    ┆ i64      ┆ str       │
╞═════════╪════════╪══════════╪═══════════╡
│ trial_9 ┆ page_9 ┆ 8        ┆ vistats   │
│ trial_9 ┆ page_9 ┆ 9        ┆ utomlands │
│ trial_9 ┆ page_9 ┆ 12       ┆ studietid │
│ trial_9 ┆ page_9 ┆ 14   

In [74]:
stimuli = sess.stimuli
stim = stimuli[0]
stim

Stimulus(id=1, name='PopSci_MultiplEYE', type='experiment', pages=[StimulusPage(number=1, text='MultiplEYE-projektet\n\nNamnet ”MultiplEYE” är en ordlek som kombinerar ”multilingualism” eller ”multiple languages” med ”eye” från ”eye-tracking”. MultiplEYE är en COST-aktion finansierad av den Europeiska unionen. COST-aktioner är forskningsnätverk som stöds av European Cooperation in Science and Technology, förkortat COST. Som finansiär stöder COST vårt växande nätverk av forskare i och utanför Europa genom att ge ekonomiskt stöd till genomförandet av olika networkingevent.', image_path=PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/stimuli_images_sv_ch_1/popsci_multipleye_id1_page_1_sv.png'), aoi_image_path=PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/aoi_stimuli_images_sv_

In [77]:
aois = stim.text_stimulus.aois
words_only = all_tokens_from_aois(aois, trial=stim.trial_id)
words_only = words_only.with_columns(
    pl.lit(stim.name).alias(settings.STIMULUS_COL)
)
trial_idx = stim.trial_id

In [79]:
words_only

trial,page,word_idx,word,stimulus
str,str,i64,str,str
"""trial_10""","""page_5""",0,"""Med""","""PopSci_MultiplEYE"""
"""trial_10""","""page_11""",0,"""De""","""PopSci_MultiplEYE"""
"""trial_10""","""page_2""",0,"""Dessa""","""PopSci_MultiplEYE"""
"""trial_10""","""page_1""",0,"""MultiplEYE-projektet""","""PopSci_MultiplEYE"""
"""trial_10""","""page_8""",0,"""Motivationen""","""PopSci_MultiplEYE"""
…,…,…,…,…
"""trial_10""","""page_9""",85,"""av""","""PopSci_MultiplEYE"""
"""trial_10""","""page_9""",86,"""nyckelord""","""PopSci_MultiplEYE"""
"""trial_10""","""page_9""",87,"""från""","""PopSci_MultiplEYE"""


In [82]:
rm_all_trials = []

for (trial_idx, stim_name, page_idx), df in only_fix.group_by(group_columns):
    words_only = all_tokens_from_aois(df, trial=trial_idx)
    rm = compute_reading_measures(
        fixations=df,
        aois=words_only,
        word_index_column=settings.WORD_IDX_COL,
        word_column="word",
    )
    rm = rm.with_columns(
        pl.lit(trial_idx).alias(settings.TRIAL_COL),
        pl.lit(page_idx).alias(settings.PAGE_COL),
        pl.lit(stim_name).alias(settings.STIMULUS_COL),
    )
    rm_all_trials.append(rm)

In [83]:
rm_df = pl.concat(rm_all_trials)
rm_df

word,word_index,FFD,SFD,FD,FPRT,FRT,TFT,RRT,RPD_inc,RPD_exc,RBRT,Fix,FPF,RR,FPReg,TRC_out,TRC_in,SL_in,SL_out,TFC,trial,page,stimulus
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str
"""mjölkprodukter""",0,199,199,199,199,199,199,0,199,0,199,1,1,0,0,0,0,1,1,1,"""trial_5""","""page_3""","""Arg_PISACowsMilk_10"""
"""innehåller""",1,194,194,194,194,194,194,0,194,0,194,1,1,0,0,0,0,1,1,1,"""trial_5""","""page_3""","""Arg_PISACowsMilk_10"""
"""viktiga""",2,209,209,209,209,209,209,0,209,0,209,1,1,0,0,0,0,1,3,1,"""trial_5""","""page_3""","""Arg_PISACowsMilk_10"""
"""kalcium,""",5,196,196,196,196,196,196,0,196,0,196,1,1,0,0,0,0,3,3,1,"""trial_5""","""page_3""","""Arg_PISACowsMilk_10"""
"""D,""",8,117,117,117,117,117,117,0,117,0,117,1,1,0,0,0,0,3,3,1,"""trial_5""","""page_3""","""Arg_PISACowsMilk_10"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""separata""",80,331,331,331,331,331,331,0,331,0,331,1,1,0,0,0,0,2,1,1,"""trial_9""","""page_3""","""Ins_LearningMobility_3"""
"""dokument""",81,213,213,213,213,213,213,0,213,0,213,1,1,0,0,0,0,1,2,1,"""trial_9""","""page_3""","""Ins_LearningMobility_3"""
"""finns""",83,170,170,170,170,170,170,0,170,0,170,1,1,0,0,0,0,2,1,1,"""trial_9""","""page_3""","""Ins_LearningMobility_3"""


## Stage 3: Calculate AOI-based Measures

In this last step, we calculate the aoi-based measures. These are also refered to as reading measures, as they are typically used in reading research. They include measures such as first pass fixation duration (FPF), total fixation count (TFC), regression path duration (RPD), and many more. These measures are calculated based on the fixations that were mapped to the AOIs in the previous step.

### Fixation-based Metrics

As an intermediate step, the fixations are annotated. These annoataions include:
- The run ID. This ID specifies continuous sequences of fixations on the same word. It is used to calculate first pass and second pass measures.
- Whether the fixation is within the first pass or not
- The index of the preceding word and the following word
- If the saccade entering or leaving the fixation is a regression or not
- Whether it is the first fixation on the word or not

This information is necessary to calculate the reading measures in the next step.

In [ ]:
rm_df = preprocessing.calculate_reading_measures(gaze, sess.stimuli)
preprocessing.save_reading_measures(sid, rm_df)

In [ ]:
rm_df.filter(pl.col("trial") == "trial_1").head()

## Stage 4: Comprehension Question Answers
In addition to gaze data, each session contains answers to comprehension questions.
These are extracted from the ASC messages. The answers are matched to the stimulus order using the `question_order_versions.csv` file in the session's logfiles folder.

In [ ]:
answers_csv = sid.answers_dir / f"{sid}_answers.csv"
question_order_csv = (
    sess.session_folder_path / "logfiles" / "question_order_versions.csv"
)
parsed_answers = preprocessing.parse_answers_from_messages(gaze.messages)
source = "asc"

In [ ]:
parsed_answers

In [ ]:
preprocessing.collect_session_answers(
    question_order_csv=question_order_csv,
    stimuli_trial_map=sess.stimuli_trial_mapping,
    stimuli=sess.stimuli,
    parsed_answers=parsed_answers,
    out_path=answers_csv,
    source=source,
    completed_stimuli_ids=sess.completed_stimuli_ids,
)

## Final Steps

In the very end, we can create the session and dataset overview and store them as well. In addition, the participant data can be parsed and stored.

For the MultiplEYE data, there is also the option to create a sanity check report.

In [ ]:
multipleye.create_sanity_check_report(
    gaze,
    sess.session_identifier,
    output_dir=settings.OUTPUT_DIR,
    plotting=True,
    overwrite=True,
)

In [ ]:
multipleye.create_session_overview(sess.session_identifier, path=settings.OUTPUT_DIR)
multipleye.create_dataset_overview(path=settings.OUTPUT_DIR)
multipleye.parse_participant_data(settings.OUTPUT_DIR / "participant_data.csv")

In [ ]:
from preprocessing.psychometric_tests.preprocess_psychometric_tests import (
    preprocess_all_sessions,
)

preprocess_all_sessions()